# Phase 4 — Affordability & Policy Engine

## Affordability Formula
```
Adjusted Income      = income_estimate × BCI haircut
Allowable Obligation = Adjusted Income × 40% (BOT DSCR)
Existing Obligations = From transaction commitment features
ADSC                 = Allowable Obligation − Existing Obligations
```

## Decision Matrix
| BCI Band | Affordable | Decision |
|---|---|---|
| HIGH / MEDIUM | YES | STP_APPROVE |
| HIGH / MEDIUM | NO | STP_DECLINE |
| LOW | YES | REFER_INCOME_VERIFY |
| LOW | NO | STP_DECLINE |
| VERY LOW | ANY | STP_DECLINE |

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.affordability import AffordabilityEngine, PolicyEngine
from src.utils import setup_logging

setup_logging('INFO')

In [ ]:
# Assumes bci_results and features_df from previous notebooks
engine = AffordabilityEngine(config_path='../config/config.yaml')
affordability_results = engine.compute(bci_results, features_df)
engine.get_summary(affordability_results)

## DSCR Stress Test — Policy Calibration

In [ ]:
stress = engine.stress_test(affordability_results, dscr_scenarios=[0.30, 0.35, 0.40, 0.45, 0.50])
print(stress)

stress.plot(x='dscr_cap', y='approval_rate', marker='o', figsize=(8, 4))
plt.title('Approval Rate vs DSCR Cap')
plt.ylabel('Approval Rate (%)')
plt.xlabel('DSCR Cap')
plt.axvline(0.40, color='red', linestyle='--', label='Current BOT norm')
plt.legend()
plt.show()

## Final Policy Decisions

In [ ]:
policy = PolicyEngine(config_path='../config/config.yaml')
final_output = policy.get_full_output(
    bci_results, affordability_results, income_results, features_df
)

# Decision summary
policy.get_decision_summary(policy.decide(bci_results, affordability_results))

In [ ]:
# Export final output (within GDZ only)
# final_output.to_parquet('../data/processed/final_decisions.parquet')
print(f'Total customers: {len(final_output):,}')
print(final_output['final_decision'].value_counts())